# Exploratory Data Analysis — Customer Churn

**Goal:** Understand the dataset, identify the strongest signals predicting churn, and build intuition before modelling.

**Dataset:** Telco Customer Churn — 7,043 customers, 21 features

**Structure:**
1. Load & first look
2. Target distribution
3. Missing values & data quality
4. Numerical features
5. Categorical features
6. Key churn signals
7. Engineered feature preview
8. Summary findings

## 1. Load & first look

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/telco-churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 2. Target distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=['#5DCAA5', '#D85A30'], width=0.5)
axes[0].set_title('Customer count by churn status')
axes[0].set_ylabel('Number of customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            colors=['#5DCAA5', '#D85A30'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn rate')

plt.suptitle('Target distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/01_target_distribution.png', bbox_inches='tight')
plt.show()

print(f'Churn rate: {churn_pct["Yes"]:.1f}%')
print(f'Class imbalance ratio: 1:{churn_counts["No"]/churn_counts["Yes"]:.1f}')

> **Finding:** The dataset has a ~26% churn rate — moderately imbalanced. 
We will use `class_weight='balanced'` in models rather than oversampling.

## 3. Missing values & data quality

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]

print('Missing values:')
print(missing if len(missing) > 0 else 'None found')
print()

# TotalCharges is coerced to NaN for new customers (tenure=0)
new_customers = df[df['TotalCharges'].isnull()]
print(f'Rows with missing TotalCharges: {len(new_customers)}')
print(f'Their tenure values: {new_customers["tenure"].unique()}')
print()
print('Strategy: impute with median (handled in pipeline)')

In [ ]:
# Drop customerID — not a feature
df = df.drop(columns=['customerID'])
print('customerID dropped. Final shape:', df.shape)

## 4. Numerical features

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

colors = {'Yes': '#D85A30', 'No': '#5DCAA5'}

for i, col in enumerate(num_cols):
    # Distribution
    for churn_val, color in colors.items():
        subset = df[df['Churn'] == churn_val][col].dropna()
        axes[0, i].hist(subset, bins=30, alpha=0.6, color=color,
                        label=f'Churn={churn_val}', density=True)
    axes[0, i].set_title(f'{col} distribution')
    axes[0, i].legend(fontsize=9)
    axes[0, i].set_xlabel(col)

    # Boxplot
    churn_no = df[df['Churn'] == 'No'][col].dropna()
    churn_yes = df[df['Churn'] == 'Yes'][col].dropna()
    axes[1, i].boxplot([churn_no, churn_yes], labels=['No Churn', 'Churn'],
                       patch_artist=True,
                       boxprops=dict(facecolor='#E1F5EE'),
                       medianprops=dict(color='#D85A30', linewidth=2))
    axes[1, i].set_title(f'{col} by churn')

plt.suptitle('Numerical features vs churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/02_numerical_features.png', bbox_inches='tight')
plt.show()

> **Findings:**
> - **Tenure:** Churners have significantly lower tenure — most leave in the first 12 months
> - **MonthlyCharges:** Churners pay more on average (~$74 vs ~$61)
> - **TotalCharges:** Lower for churners — consistent with shorter tenure

## 5. Categorical features

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod',
            'TechSupport', 'OnlineSecurity', 'PaperlessBilling']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).sort_values(ascending=True)

    bars = axes[i].barh(churn_rate.index, churn_rate.values,
                        color=['#D85A30' if v > 26 else '#5DCAA5' for v in churn_rate.values])
    axes[i].axvline(x=26, color='gray', linestyle='--', linewidth=1, label='Avg churn rate')
    axes[i].set_title(f'Churn rate by {col}')
    axes[i].set_xlabel('Churn rate (%)')
    axes[i].xaxis.set_major_formatter(mtick.PercentFormatter())

    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}%', va='center', fontsize=9)

plt.suptitle('Churn rate by categorical feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/03_categorical_features.png', bbox_inches='tight')
plt.show()

> **Findings:**
> - **Contract:** Month-to-month customers churn at ~43% vs ~3% for two-year contracts — the strongest single signal
> - **InternetService:** Fiber optic customers churn at ~42% — possibly due to higher prices
> - **TechSupport / OnlineSecurity:** No-support customers churn at 2x the rate
> - **PaperlessBilling:** Correlated with higher churn — likely proxy for digital-native, price-sensitive customers

## 6. Key churn signals

In [ ]:
# Tenure buckets — where do customers churn?
df['tenure_bucket'] = pd.cut(df['tenure'],
    bins=[0, 6, 12, 24, 48, 72],
    labels=['0-6m', '6-12m', '12-24m', '24-48m', '48-72m'])

tenure_churn = df.groupby('tenure_bucket', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by tenure bucket
bars = axes[0].bar(tenure_churn.index, tenure_churn.values,
                   color=['#D85A30' if v > 26 else '#5DCAA5' for v in tenure_churn.values])
axes[0].axhline(y=26, color='gray', linestyle='--', linewidth=1, label='Avg churn rate')
axes[0].set_title('Churn rate by tenure bucket')
axes[0].set_xlabel('Tenure')
axes[0].set_ylabel('Churn rate (%)')
axes[0].legend()
for bar, val in zip(bars, tenure_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Contract x MonthlyCharges heatmap proxy
high_risk = df[
    (df['Contract'] == 'Month-to-month') &
    (df['tenure'] <= 6)
].copy()
high_risk_churn = (high_risk['Churn'] == 'Yes').mean() * 100

segments = {
    'Month-to-month\n+ tenure ≤6m': high_risk_churn,
    'Month-to-month\n+ tenure >6m': (df[(df['Contract']=='Month-to-month') & (df['tenure']>6)]['Churn']=='Yes').mean()*100,
    'One year\ncontract': (df[df['Contract']=='One year']['Churn']=='Yes').mean()*100,
    'Two year\ncontract': (df[df['Contract']=='Two year']['Churn']=='Yes').mean()*100,
}

bars2 = axes[1].bar(segments.keys(), segments.values(),
                    color=['#D85A30', '#F0997B', '#9FE1CB', '#5DCAA5'])
axes[1].axhline(y=26, color='gray', linestyle='--', linewidth=1, label='Avg churn rate')
axes[1].set_title('Churn rate by contract + tenure segment')
axes[1].set_ylabel('Churn rate (%)')
axes[1].legend()
for bar, val in zip(bars2, segments.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('High-risk customer segments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/04_key_signals.png', bbox_inches='tight')
plt.show()

print(f'Highest risk segment (month-to-month + tenure ≤6m): {high_risk_churn:.1f}% churn rate')
print(f'That is {high_risk_churn/26:.1f}x the average churn rate')

## 7. Engineered feature preview

In [ ]:
# charges_per_tenure = MonthlyCharges / (tenure + 1)
df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for churn_val, color in colors.items():
    subset = df[df['Churn'] == churn_val]['charges_per_tenure']
    axes[0].hist(subset, bins=40, alpha=0.6, color=color,
                 label=f'Churn={churn_val}', density=True)
axes[0].set_title('charges_per_tenure distribution')
axes[0].set_xlabel('MonthlyCharges / (tenure + 1)')
axes[0].legend()

churn_no = df[df['Churn'] == 'No']['charges_per_tenure']
churn_yes = df[df['Churn'] == 'Yes']['charges_per_tenure']
axes[1].boxplot([churn_no, churn_yes], labels=['No Churn', 'Churn'],
                patch_artist=True,
                boxprops=dict(facecolor='#E1F5EE'),
                medianprops=dict(color='#D85A30', linewidth=2))
axes[1].set_title('charges_per_tenure by churn')

plt.suptitle('Engineered feature: charges_per_tenure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/05_engineered_feature.png', bbox_inches='tight')
plt.show()

print(f'Mean charges_per_tenure — No churn: {churn_no.mean():.2f}')
print(f'Mean charges_per_tenure — Churn:    {churn_yes.mean():.2f}')
print(f'Ratio: {churn_yes.mean()/churn_no.mean():.1f}x higher for churners')

> **Finding:** `charges_per_tenure` is ~3x higher for churners — it captures the combined effect 
of high charges and short tenure in a single feature. This will be valuable for the model.

## 8. Summary findings

In [ ]:
print('=' * 55)
print('EDA SUMMARY — TOP CHURN SIGNALS')
print('=' * 55)

signals = [
    ('Contract type', 'Month-to-month = 43% churn vs 3% for two-year'),
    ('Tenure',        'First 6 months = highest risk window'),
    ('charges_per_tenure', 'Churners are 3x higher than non-churners'),
    ('InternetService', 'Fiber optic = 42% churn rate'),
    ('TechSupport',   'No support = 2x average churn rate'),
]

for i, (feature, insight) in enumerate(signals, 1):
    print(f'{i}. {feature}')
    print(f'   → {insight}')
    print()

print('Class imbalance: 26% churn — handle with class_weight="balanced"')
print('Missing values:  11 rows in TotalCharges — impute with median')
print('Next step:       notebooks/02_modelling.ipynb')